# Notebook 03: Knowledge Graph Construction & Topological Intelligence

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 03 of 12  
**Focus:** NetworkX Property Graph, Entity Resolution, Focused Subgraphs, Topological Validation  

---

## 1. Business Problem

Relational databases (SQL/DuckDB) represent enterprise architectures as separate tables linked by foreign keys. While effective for tabular aggregations, relational joins degrade rapidly in readability and performance when answering **multi-hop structural questions**:
- *Which business units depend on applications running on technologies supplied by Vendor X?*
- *If Application Y is retired, what upstream applications, IT services, and business capabilities will experience disruptions?*

In SQL, answering an 8-hop relationship query requires chaining 7 table joins, nested subqueries, and recursive Common Table Expressions (CTEs). 

A **Knowledge Graph** models entities as first-class nodes and business connections as first-class directed edges, transforming complex multi-join relational queries into intuitive graph path traversals.

---

## 2. Core Concept & Formulation

### 2.1 Property Graph Model
We construct a directed multigraph $G = (V, E)$ where:
- Each vertex $v \in V$ possesses a unique identifier, an `entity_type`, and domain properties (e.g., `lifecycle_status`, `criticality`, `annual_license`).
- Each directed edge $e = (u, v) \in E$ possesses a typed `relationship` attribute (e.g., `SUPPORTS`, `RUNS_ON`, `OWNS`, `DEPENDS_ON`).

### 2.2 Overcoming the "Hairball" Problem
A major failure mode in enterprise graph analytics is visualizing all 500+ nodes and 1,000+ relationships simultaneously on a single canvas, resulting in an unreadable visual "hairball". 

We employ **focused subgraph ego-networks**:
$$G_{\text{sub}}(v_0, r) = \{ v \in V \mid \text{dist}(v_0, v) \le r \}$$
By bounding visualization to radius $r \in \{1, 2\}$ around a focal entity, we deliver crisp, decision-ready topological insights.

In [1]:
import sys
from pathlib import Path
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config, get_project_root
from tvi.graph import (
    build_knowledge_graph, validate_graph, save_graph, get_focused_subgraph,
    _add_business_units, _add_capabilities, _add_services, _add_applications,
    _add_technologies, _add_vendors, _add_projects, _add_benefits, _add_kpis
)
from tvi.graph_queries import find_business_units_by_vendor, find_capability_overlapping_apps

config = load_config()
source_dir = get_project_root() / "data" / "generated"

# Construct graph progressively using modular builder functions
G = nx.DiGraph(name="TechnologyValueIntelligence")

# Load tables
bu_df = pd.read_csv(source_dir / "business_units.csv")
cap_df = pd.read_csv(source_dir / "business_capabilities.csv")
srv_df = pd.read_csv(source_dir / "it_services.csv")
app_df = pd.read_csv(source_dir / "applications.csv")
tech_df = pd.read_csv(source_dir / "technologies.csv")
ven_df = pd.read_csv(source_dir / "vendors.csv")
prj_df = pd.read_csv(source_dir / "projects.csv")
ben_df = pd.read_csv(source_dir / "benefits.csv")
kpi_df = pd.read_csv(source_dir / "kpis.csv")

# Add nodes progressively
_add_business_units(G, bu_df)
_add_capabilities(G, cap_df)
_add_services(G, srv_df)
_add_applications(G, app_df)
_add_technologies(G, tech_df)
_add_vendors(G, ven_df)
_add_projects(G, prj_df)
_add_benefits(G, ben_df)
_add_kpis(G, kpi_df)

print(f"Total Nodes Added: {G.number_of_nodes()}")

Total Nodes Added: 283


In [2]:
# Add Multi-Relational Edges
rel_app_tech = pd.read_csv(source_dir / "rel_app_technology.csv")
rel_app_cap = pd.read_csv(source_dir / "rel_app_capability.csv")
rel_app_dep = pd.read_csv(source_dir / "rel_app_dependency.csv")
rel_prj_cap = pd.read_csv(source_dir / "rel_project_capability.csv")
rel_app_prj = pd.read_csv(source_dir / "rel_app_project.csv")
rel_ben_kpi = pd.read_csv(source_dir / "rel_benefit_kpi.csv")

# BU OWNS Capability
for _, r in cap_df.iterrows():
    G.add_edge(r["business_unit_id"], r["capability_id"], relationship="OWNS")

# App SUPPORTS Capability
for _, r in rel_app_cap.iterrows():
    G.add_edge(r["application_id"], r["capability_id"], relationship="SUPPORTS", support_type=r.get("support_type", "Primary"))

# Service and Application links
for _, r in app_df.iterrows():
    G.add_edge(r["service_id"], r["application_id"], relationship="DELIVERED_BY")
    G.add_edge(r["application_id"], r["service_id"], relationship="SUPPORTS_SERVICE")

# App RUNS_ON Technology
for _, r in rel_app_tech.iterrows():
    G.add_edge(r["application_id"], r["technology_id"], relationship="RUNS_ON")

# Tech SUPPLIED_BY Vendor
for _, r in tech_df.iterrows():
    m = ven_df[ven_df["vendor_name"] == r["technology_vendor"]]
    if not m.empty:
        G.add_edge(r["technology_id"], m.iloc[0]["vendor_id"], relationship="SUPPLIED_BY")

# App DEPENDS_ON App
for _, r in rel_app_dep.iterrows():
    G.add_edge(r["source_app_id"], r["target_app_id"], relationship="DEPENDS_ON", criticality=r.get("criticality", "High"))

# Project Links
for _, r in rel_prj_cap.iterrows():
    G.add_edge(r["project_id"], r["capability_id"], relationship="TARGETS")

for _, r in rel_app_prj.iterrows():
    G.add_edge(r["project_id"], r["application_id"], relationship="FUNDS_MODERNIZATION")

for _, r in ben_df.iterrows():
    G.add_edge(r["project_id"], r["benefit_id"], relationship="PRODUCES")

for _, r in rel_ben_kpi.iterrows():
    G.add_edge(r["benefit_id"], r["kpi_id"], relationship="MEASURED_BY")

print(f"Total Graph Constructed: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Edges")

Total Graph Constructed: 283 Nodes, 675 Edges


---

## 3. Quantitative Topological & Centrality Analysis

We compute topological network metrics to identify key structural hubs:
- **Degree Centrality:** High degree indicates critical enterprise dependencies.
- **In-Degree vs Out-Degree:** Identifies bottleneck providers (high in-degree) vs heavy consumers (high out-degree).

In [3]:
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

# Filter to Applications
app_degrees = []
for a_id in app_df["application_id"]:
    app_degrees.append({
        "application_id": a_id,
        "application_name": G.nodes[a_id].get("label", a_id),
        "in_degree": in_degrees.get(a_id, 0),
        "out_degree": out_degrees.get(a_id, 0),
        "total_degree": in_degrees.get(a_id, 0) + out_degrees.get(a_id, 0),
        "lifecycle": G.nodes[a_id].get("lifecycle_status", "")
    })

app_deg_df = pd.DataFrame(app_degrees).sort_values("in_degree", ascending=False)
print("Top 5 Critical Hub Applications by Inbound Dependency:")
display(app_deg_df.head(5))

# Assert Scenario D hub: APP010 must be the highest in-degree app
assert app_deg_df.iloc[0]["application_id"] == "APP010", "APP010 must be the highest in-degree dependency hub!"

Top 5 Critical Hub Applications by Inbound Dependency:


---

## 4. Focused Subgraph Visualizations

We extract and visualize bounded ego-networks around focal entities:
1. **Application Ego-Network (`APP001` - Scale Anchor)**
2. **Central Auth Hub Ego-Network (`APP010` - Single Point of Failure)**

In [4]:
# Focused Subgraph: APP001
sub_g1 = get_focused_subgraph(G, "APP001", radius=1, max_nodes=25)

plt.figure(figsize=(9, 5.5))
pos = nx.spring_layout(sub_g1, seed=42)

colors = []
for n in sub_g1.nodes():
    t = sub_g1.nodes[n].get("entity_type", "")
    if t == "Application": colors.append("#d62728")
    elif t == "Technology": colors.append("#1f77b4")
    elif t == "BusinessCapability": colors.append("#2ca02c")
    elif t == "ITService": colors.append("#ff7f0e")
    else: colors.append("#9467bd")

nx.draw_networkx_nodes(sub_g1, pos, node_color=colors, node_size=1300, alpha=0.9)
nx.draw_networkx_edges(sub_g1, pos, edge_color="#666666", arrows=True, arrowsize=15, width=1.5)
labels = {n: sub_g1.nodes[n].get("label", n) for n in sub_g1.nodes()}
nx.draw_networkx_labels(sub_g1, pos, labels=labels, font_size=8, font_weight="bold")

plt.title("Focused Knowledge Subgraph: APP001 (CoreBanking Alpha)", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()

In [5]:
# Interactive Plotly Network Visualization for APP010 (Scenario D Hub)
sub_g2 = get_focused_subgraph(G, "APP010", radius=1, max_nodes=25)
pos2 = nx.spring_layout(sub_g2, seed=42)

edge_x = []
edge_y = []
for edge in sub_g2.edges():
    x0, y0 = pos2[edge[0]]
    x1, y1 = pos2[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=1.5, color="#888"),
    hoverinfo="none",
    mode="lines"
)

node_x = []
node_y = []
node_text = []
node_color = []
for node in sub_g2.nodes():
    x, y = pos2[node]
    node_x.append(x)
    node_y.append(y)
    data = sub_g2.nodes[node]
    node_text.append(f"{node}: {data.get('label', '')}<br>Type: {data.get('entity_type', '')}")
    node_color.append("#d62728" if node == "APP010" else "#1f77b4")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=[sub_g2.nodes[n].get("label", n)[:12] for n in sub_g2.nodes()],
    textposition="top center",
    hovertext=node_text,
    hoverinfo="text",
    marker=dict(
        color=node_color,
        size=22,
        line_width=2,
        line_color="black"
    )
)

fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title="Interactive Topology: APP010 Central Auth Hub & Inbound Dependencies",
                showlegend=False,
                hovermode="closest",
                margin=dict(b=20, l=5, r=5, t=40),
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
             )
fig.show()

---

## 5. Multi-Hop Traversal Demonstration

We execute a real-world multi-hop query:
> *"Which business units depend on applications running on technologies supplied by Vendor X?"*

Traversal: `Vendor` $\longrightarrow$ `Technology` $\longrightarrow$ `Application` $\longrightarrow$ `Capability` $\longrightarrow$ `BusinessUnit`

In [6]:
vendor_results = find_business_units_by_vendor(G, "TechNova")
v_df = pd.DataFrame(vendor_results)
print(f"Total Traversal Paths Found for Vendor 'TechNova': {len(v_df)}")
display(v_df[["vendor_name", "technology_name", "application_name", "capability_name", "business_unit_name"]].head(6))

Total Traversal Paths Found for Vendor 'TechNova': 42


## 6. Limitations & Methodological Guardrails

1. **In-Memory RAM Limits:** NetworkX stores the entire graph in Python memory. This is ideal for 16 GB machines up to ~100k nodes, but enterprise deployments with 10M+ nodes require Neo4j or Amazon Neptune.
2. **Dynamic Frequency:** Graph edges represent structural connections; edge traversal does not measure call throughput per second without attaching dynamic telemetry.

## 7. Next Step

In **Notebook 04: Application Cost Intelligence**, we combine the relational cost ledgers with consumption telemetry to compute true fully-burdened application TCO and unit economics.